# HydraNet Multi-Head Component Export

This notebook mirrors the component export flow from `notebooks/start_here.ipynb` and changes only the decoder multiplicity.

It:
1. Loads a student checkpoint-compatible model.
2. Splits it into encoder, bottleneck, and decoder components.
3. Exports encoder and bottleneck exactly as before.
4. Replaces the single decoder export with a configurable multi-head decoder export where all heads consume the same bottleneck embedding and skip tensors.

Default configuration exports `5` decoder heads, but `num_heads` is parameterized.

## 1. Parameters And Model Loading

Set `num_heads` to the number of decoder branches you want to export. Each branch is initialized as a deep copy of the original decoder stack, so all heads start with the same weights and consume the same embedding input.

In [1]:
from pathlib import Path

import hydranet

# Model loading parameters
task = 'burned_area'
n_shots = 5000
training = 'finetuning'
weights_dir = '../weights'
auto_load_weights = True
checkpoint_selection = 'best'
strict = False

# Multi-head export parameters
num_heads = 5
input_size = 224
head_output_prefix = 'decoder_head'
output_dir = Path('../onnx/components_multihead')
output_dir.mkdir(parents=True, exist_ok=True)

model = hydranet.load_student(
    preset='checkpoint',
    task=task,
    n_shots=n_shots,
    training=training,
    auto_load_weights=auto_load_weights,
    checkpoint_selection=checkpoint_selection,
    weights_dir=weights_dir,
    strict=strict,
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Number of decoder heads to export: {num_heads}')

Selected best checkpoint using artifacts metrics: 20251216 (mcc_macro=0.8547696133029593, f1_macro=0.8900467581281862, f1=None, acc=0.8826103415975766, best_val_loss=0.2482508525252342)
  Downloading: UNet_Myriad2_Downstream_unfrozen_best.pt


  Saved to: ../weights/finetuning/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_unfrozen_5000/UNet_Myriad2_Downstream_unfrozen_best.pt
Loading weights from: ../weights/finetuning/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_unfrozen_5000/UNet_Myriad2_Downstream_unfrozen_best.pt
  Adjusted n_classes to checkpoint head: 4
Model parameters: 351,172
Number of decoder heads to export: 5


## 2. Split Model Into Components

In [2]:
components = hydranet.split_model(model)
print(components)

encoder = components.encoder
bottleneck = components.bottleneck
decoder = components.decoder

print('ENCODER:', list(encoder.keys()))
print('BOTTLENECK:', list(bottleneck.keys()))
print('DECODER:', list(decoder.keys()))

ModelComponents(
  ENCODER: 2 layer groups, 52,304 parameters (14.9%)
  BOTTLENECK: 1 layer groups, 146,816 parameters (41.8%)
  DECODER: 3 layer groups, 152,052 parameters (43.3%)
  TOTAL: 351,172 parameters
)
ENCODER: ['encoders', 'pools']
BOTTLENECK: ['bottleneck']
DECODER: ['upsamplers', 'decoders', 'final_conv']


## 3. Export Encoder, Bottleneck, And Multi-Head Decoder To ONNX

The encoder and bottleneck exports match the existing notebook. Only the decoder wrapper changes: it now returns `num_heads` outputs while keeping the same input contract and opset.

In [3]:
import copy

import torch
import torch.nn as nn

model.eval()
input_channels = int(model.n_channels)


class EncoderExportWrapper(nn.Module):
    def __init__(self, encoders, pools):
        super().__init__()
        self.encoders = encoders
        self.pools = pools

    def forward(self, x):
        skips = []
        current = x
        for index, encoder_block in enumerate(self.encoders):
            current = encoder_block(current)
            skips.append(current)
            if index < len(self.encoders) - 1:
                current = self.pools[index](current)
        bottleneck_in = self.pools[-1](current)
        return (bottleneck_in, *skips)


class DecoderHead(nn.Module):
    def __init__(self, upsamplers, decoders, final_conv):
        super().__init__()
        self.upsamplers = copy.deepcopy(upsamplers)
        self.decoders = copy.deepcopy(decoders)
        self.final_conv = copy.deepcopy(final_conv)

    def forward(self, bottleneck_out, skips):
        current = bottleneck_out
        depth = len(self.decoders)
        for index in range(depth):
            current = self.upsamplers[index](current)
            skip = skips[depth - 1 - index]
            # Keep opset 10 export free of Resize/Upsample: this model at 224x224 already matches skip sizes.
            current = torch.cat([current, skip], dim=1)
            current = self.decoders[index](current)
        return self.final_conv(current)


class MultiHeadDecoderExportWrapper(nn.Module):
    def __init__(self, upsamplers, decoders, final_conv, num_heads):
        super().__init__()
        if num_heads < 1:
            raise ValueError('num_heads must be at least 1')
        self.num_heads = int(num_heads)
        self.heads = nn.ModuleList(
            [DecoderHead(upsamplers, decoders, final_conv) for _ in range(self.num_heads)]
        )

    def forward(self, bottleneck_out, *skips):
        skip_list = list(skips)
        return tuple(head(bottleneck_out, skip_list) for head in self.heads)


encoder_wrapper = EncoderExportWrapper(encoder['encoders'], encoder['pools']).eval()
bottleneck_wrapper = bottleneck['bottleneck'].eval()
multi_head_decoder_wrapper = MultiHeadDecoderExportWrapper(
    decoder['upsamplers'],
    decoder['decoders'],
    decoder['final_conv'],
    num_heads=num_heads,
).eval()

dummy_input = torch.randn(1, input_channels, input_size, input_size)

with torch.no_grad():
    encoder_outputs = encoder_wrapper(dummy_input)
    bottleneck_input = encoder_outputs[0]
    skips = encoder_outputs[1:]
    bottleneck_output = bottleneck_wrapper(bottleneck_input)
    decoder_outputs = multi_head_decoder_wrapper(bottleneck_output, *skips)

assert len(decoder_outputs) == num_heads
expected_decoder_shape = tuple(decoder_outputs[0].shape)
for output in decoder_outputs:
    assert tuple(output.shape) == expected_decoder_shape

component_paths = {
    'encoder': output_dir / 'encoder.onnx',
    'bottleneck': output_dir / 'bottleneck.onnx',
    'multi_head_decoder': output_dir / f'decoder_{num_heads}_heads.onnx',
}

# 1) Encoder export: output = bottleneck_input + all skip tensors
torch.onnx.export(
    encoder_wrapper,
    dummy_input,
    str(component_paths['encoder']),
    export_params=True,
    opset_version=10,
    input_names=['input'],
    output_names=['bottleneck_input'] + [f'skip_{index}' for index in range(len(skips))],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'bottleneck_input': {0: 'batch_size'},
        **{f'skip_{index}': {0: 'batch_size'} for index in range(len(skips))},
    },
)

# 2) Bottleneck export: input = bottleneck_input, output = bottleneck_output
torch.onnx.export(
    bottleneck_wrapper,
    bottleneck_input,
    str(component_paths['bottleneck']),
    export_params=True,
    opset_version=10,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
)

# 3) Multi-head decoder export: inputs = bottleneck_output + skip tensors, outputs = one tensor per decoder head
decoder_input_names = ['bottleneck_output'] + [f'skip_{index}' for index in range(len(skips))]
decoder_output_names = [f'{head_output_prefix}_{index}' for index in range(num_heads)]
decoder_dynamic_axes = {name: {0: 'batch_size'} for name in decoder_input_names}
decoder_dynamic_axes.update({name: {0: 'batch_size'} for name in decoder_output_names})

torch.onnx.export(
    multi_head_decoder_wrapper,
    (bottleneck_output, *skips),
    str(component_paths['multi_head_decoder']),
    export_params=True,
    opset_version=10,
    input_names=decoder_input_names,
    output_names=decoder_output_names,
    dynamic_axes=decoder_dynamic_axes,
)

print(f'Per-head output shape: {expected_decoder_shape}')
for name, path in component_paths.items():
    print(f'{name}: {path.resolve()}')

Per-head output shape: (1, 4, 224, 224)
encoder: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/onnx/components_multihead/encoder.onnx
bottleneck: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/onnx/components_multihead/bottleneck.onnx
multi_head_decoder: /shared/home/rdelprete/PythonProjects/hydranet-phisat2/onnx/components_multihead/decoder_5_heads.onnx


## 4. Validate The Exported Multi-Head ONNX Graph

In [4]:
from collections import Counter

import onnx
from onnx import shape_inference


def tensor_shape_metadata(value_info):
    dims = []
    for dim in value_info.type.tensor_type.shape.dim:
        if dim.HasField('dim_value'):
            dims.append(dim.dim_value)
        elif dim.HasField('dim_param'):
            dims.append(dim.dim_param)
        else:
            dims.append('?')
    return dims


decoder_onnx = onnx.load(str(component_paths['multi_head_decoder']))
onnx.checker.check_model(decoder_onnx)
shape_metadata_source = 'graph output metadata'
shape_metadata_graph = decoder_onnx.graph
try:
    shape_metadata_graph = shape_inference.infer_shapes(decoder_onnx).graph
    shape_metadata_source = 'shape_inference output metadata'
except Exception as exc:
    print(f'Shape inference unavailable, falling back to graph metadata: {exc}')
decoder_opset = max(
    entry.version for entry in decoder_onnx.opset_import if entry.domain in ('', 'ai.onnx')
)
exported_input_names = [input_value.name for input_value in decoder_onnx.graph.input]
exported_output_names = [output.name for output in decoder_onnx.graph.output]
exported_output_shapes = {
    output.name: tensor_shape_metadata(output)
    for output in shape_metadata_graph.output
}
expected_output_shape_metadata = ['batch_size', *expected_decoder_shape[1:]]
op_histogram = dict(sorted(Counter(node.op_type for node in decoder_onnx.graph.node).items()))

assert decoder_opset == 10, decoder_opset
assert exported_input_names == decoder_input_names, (exported_input_names, decoder_input_names)
assert len(exported_output_names) == num_heads, exported_output_names
assert exported_output_names == decoder_output_names, (exported_output_names, decoder_output_names)
for output_name, output_shape in exported_output_shapes.items():
    assert len(output_shape) == len(expected_output_shape_metadata), (
        output_name,
        output_shape,
        expected_output_shape_metadata,
    )
    for actual_dim, expected_dim in zip(output_shape, expected_output_shape_metadata):
        if actual_dim != '?':
            assert actual_dim == expected_dim, (
                output_name,
                output_shape,
                expected_output_shape_metadata,
            )

print('ONNX checker: PASS')
print('Structural validation: PASS (runtime validation is optional and requires onnxruntime)')
print(f'Decoder opset: {decoder_opset}')
print(f'Decoder inputs: {exported_input_names}')
print(f'Decoder outputs: {exported_output_names}')
print(f'Output shape metadata source: {shape_metadata_source}')
print(f'Decoder output shapes: {exported_output_shapes}')
print(f'Graph nodes: {len(decoder_onnx.graph.node)}')
print(f'Graph initializers: {len(decoder_onnx.graph.initializer)}')
print(f'Op histogram: {op_histogram}')

ONNX checker: PASS
Structural validation: PASS (runtime validation is optional and requires onnxruntime)
Decoder opset: 10
Decoder inputs: ['bottleneck_output', 'skip_0', 'skip_1', 'skip_2']
Decoder outputs: ['decoder_head_0', 'decoder_head_1', 'decoder_head_2', 'decoder_head_3', 'decoder_head_4']
Output shape metadata source: shape_inference output metadata
Decoder output shapes: {'decoder_head_0': ['batch_size', 4, 224, 224], 'decoder_head_1': ['batch_size', 4, 224, 224], 'decoder_head_2': ['batch_size', 4, 224, 224], 'decoder_head_3': ['batch_size', 4, 224, 224], 'decoder_head_4': ['batch_size', 4, 224, 224]}
Graph nodes: 385
Graph initializers: 35
Op histogram: {'Add': 30, 'Concat': 15, 'Constant': 45, 'Conv': 65, 'ConvTranspose': 15, 'Div': 15, 'Erf': 15, 'Identity': 140, 'Mul': 45}


In [5]:
import importlib.util

if importlib.util.find_spec('onnxruntime') is None:
    print('Skipping onnxruntime debug: package not installed.')
else:
    try:
        bottleneck_output_np = bottleneck_output.detach().cpu().numpy()
        skip_arrays = [skip.detach().cpu().numpy() for skip in skips]
        pytorch_decoder_outputs = [output.detach().cpu().numpy() for output in decoder_outputs]
    except NameError as exc:
        print(f'Skipping onnxruntime debug: decoder sample tensors are unavailable in this session ({exc}).')
    else:
        import numpy as np
        import onnxruntime as ort

        ort_session = ort.InferenceSession(
            str(component_paths['multi_head_decoder']),
            providers=['CPUExecutionProvider'],
        )
        ort_input_names = [input_value.name for input_value in ort_session.get_inputs()]
        ort_output_names = [output_value.name for output_value in ort_session.get_outputs()]
        ort_feeds = {'bottleneck_output': bottleneck_output_np}
        ort_feeds.update(
            {f'skip_{index}': skip_array for index, skip_array in enumerate(skip_arrays)}
        )
        ort_outputs = ort_session.run(ort_output_names, ort_feeds)
        ort_output_shapes = [tuple(output.shape) for output in ort_outputs]

        assert ort_input_names == decoder_input_names, (ort_input_names, decoder_input_names)
        assert ort_output_names == decoder_output_names, (ort_output_names, decoder_output_names)
        assert len(ort_outputs) == num_heads, len(ort_outputs)
        assert all(shape == expected_decoder_shape for shape in ort_output_shapes), (
            ort_output_shapes,
            expected_decoder_shape,
        )
        for index, (ort_output, pytorch_output) in enumerate(
            zip(ort_outputs, pytorch_decoder_outputs)
        ):
            assert np.allclose(ort_output, pytorch_output, rtol=1e-4, atol=1e-5), (
                index,
                float(np.max(np.abs(ort_output - pytorch_output))),
            )

        print('onnxruntime debug: PASS')
        print(f'onnxruntime inputs: {ort_input_names}')
        print(f'onnxruntime outputs: {ort_output_names}')
        print(f'onnxruntime output shapes: {ort_output_shapes}')

Skipping onnxruntime debug: package not installed.
